In [3]:
# Note any changes in prompt will require kernel restart

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
from agent.agent import pet_graph
from agent.utils import _print_event
import uuid
#print("DEBUG: ANTHROPIC_API_KEY =", os.getenv("ANTHROPIC_API_KEY"))

Graph successfully built.


In [2]:
def test_interactive_session():
    """
    Starts an interactive testing session with the AI agent using LangGraph's checkpointing.
    State is preserved automatically via thread_id using MemorySaver.
    """
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    printed_messages = set()
    print("🧠 Testing session started. Type 'exit' to stop.")

    while True:
        user_input = input("User: ")
        if user_input.lower() in ["exit", "quit"]:
            print("👋 Exiting session.")
            break

        message_payload = {"messages": [("user", user_input)]}
        events = pet_graph.stream(message_payload, config, stream_mode="values")
        for event in events:
            _print_event(event, printed_messages)

In [4]:
def interactive_session_clean():
    """
    A streamlined interactive loop where only the user's input and the final assistant reply are printed.
    Preserves state and uses the assistant's final message from the LangGraph state['messages'].
    """
    conversation_state = {"messages": []}
    config = {"configurable": {"user_info": "", "thread_id": str(uuid.uuid4())}}

    print("🧠 Clean Interactive Session started. Type 'exit' or 'quit' to stop.")

    while True:
        user_input = input("User: ")
        if user_input.strip().lower() in ["exit", "quit", "bye", "q"]:
            print("👋 Exiting clean interactive session.")
            break

        # Append user message
        conversation_state["messages"].append(("user", user_input))

        # Run graph and get result
        result = pet_graph.invoke(conversation_state, config)

        # Extract final assistant message
        messages = result.get("messages", [])
        if messages:
            last_msg = messages[-1]
            if hasattr(last_msg, "content"):
                final_text = last_msg.content
            elif isinstance(last_msg, dict) and "content" in last_msg:
                final_text = last_msg["content"]
            else:
                final_text = str(last_msg)
        else:
            final_text = "[No response from assistant]"

        print("Assistant:", final_text)
        print("-" * 60)

In [3]:
if __name__ == "__main__":
    interactive_session_clean()

🧠 Clean Interactive Session started. Type 'exit' or 'quit' to stop.


User:  hi


Assistant: Hello! 👋  
Welcome to **Smart Pet Travel** — your one-stop service for relocating pets safely and comfortably. 🐾

I can help you with:
- ✈️ Booking ground transportation for your pet (origin & destination)
- 💼 Getting freight quotes for pet air travel  
- 📄 Answering pet relocation FAQs and documentation requirements

To get started, could you please share your **email address** so I can look up your travel profile?
------------------------------------------------------------


User:  user3@mail.com


Assistant: Okay, let me look up the travel details for hiuser3@mail.com.

📋 Pet Travel Details
- 🐾 Name: Buddy (Labrador Retriever) 
- ✈️ Route: New York, USA → London, UK
- 📦 Dimensions: 30 x 20 x 24 inches, Weight: 50 lbs

Great, I have your pet's profile information. Would you like me to book ground transportation for you from your origin city of New York to the airport?
------------------------------------------------------------


...

Assistant: Okay, let me calculate an airline freight quote for Buddy's relocation from New York to London.

**Airline Freight Quote**
- ✈️ Route: New York → London
- 🐾 Crate: Medium (Series 300), Dimensions: 80 x 60 x 50 cm  
- 💰 Charges:
    - Check-in Fee: $300
    - Oversized Crate Fee: $150
- **Estimated Total**: $450

*Disclaimer: This quote is based on current data and airline policies. Please contact Smart Pet Travel customer care to confirm final costs and eligibility.*

Does this freight quote look okay to you? Let me know if you have any other questions!
------------------------------------------------------------


User:  Looks good


Assistant: Great, I'm glad the airline freight quote for $450 looks good to you.

Since we've covered the key details of Buddy's relocation, is there anything else I can assist you with today? I'm happy to answer any other questions you may have about the pet travel process.

If not, please let me know if you'd like me to proceed with booking the flight and ground transportation, or if you need some time to review the information.
------------------------------------------------------------


User:  Bye


👋 Exiting clean interactive session.


In [ ]:
#interactive_session_clean()